In [37]:

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import spearmanr, chi2_contingency

data = pd.read_csv("US_flights_2023.csv")

data_wt = pd.read_csv("weather_meteo_by_airport.csv")

data_cancel = pd.read_csv("Cancelled_Diverted_2023.csv")

data_geo = pd.read_csv("airports_geolocation.csv")




In [38]:
first_row_df = data.head(0)

print(first_row_df)

first_row_df_wt = data_wt.head(0)

print(first_row_df_wt)


Empty DataFrame
Columns: [FlightDate, Day_Of_Week, Airline, Tail_Number, Dep_Airport, Dep_CityName, DepTime_label, Dep_Delay, Dep_Delay_Tag, Dep_Delay_Type, Arr_Airport, Arr_CityName, Arr_Delay, Arr_Delay_Type, Flight_Duration, Distance_type, Delay_Carrier, Delay_Weather, Delay_NAS, Delay_Security, Delay_LastAircraft, Manufacturer, Model, Aicraft_age]
Index: []

[0 rows x 24 columns]
Empty DataFrame
Columns: [time, tavg, tmin, tmax, prcp, snow, wdir, wspd, pres, airport_id]
Index: []


Tương quan giữa Tuổi máy bay và Độ trễ

In [39]:
df = data.dropna(subset=["Aicraft_age", "Dep_Delay"])

df = df.sample(1000000, random_state=42)

r, pval = spearmanr(df["Aicraft_age"], df["Dep_Delay"])

print(f"Hệ số tương quan Spearman r = {r:.3f}, p-value = {pval:.5f}")

Hệ số tương quan Spearman r = -0.015, p-value = 0.00000


Có ý nghĩa thống kê, không có ý nghĩa thực tiễn (yếu , |r| ≤ 0.1) 
Trong vận hành, việc thay thế máy bay cũ bằng máy bay mới sẽ không giúp cải thiện đáng kể tình trạng trễ chuyến. Các yếu tố như bảo trì, quản lý lịch trình, và thời tiết quan trọng hơn nhiều so với tuổi của phương tiện.

MODEL MÁY BAY ẢNH HƯỞNG ĐẾN TỶ LỆ TRỄ

In [40]:
df = data.copy()
df = df.dropna(subset=["Model", "Dep_Delay_Tag"])

contingency = pd.crosstab(df["Model"], df["Dep_Delay_Tag"])
chi2, pval, dof, expected = chi2_contingency(contingency)

print("Bảng tần suất:")
print(contingency)
print(f"\nChi-square statistic = {chi2:.3f}, p-value = {pval:.5f}")

Bảng tần suất:
Dep_Delay_Tag        0        1
Model                          
135/145           8688     2464
170/175         662752   200586
190/195          45487    29775
717             129419    55500
737               2302     1248
737 NG         1523894  1179600
737 NG/MAX        6768     6808
757              88183    67964
767              11218     8987
777              13262    14593
787               7836     4738
A220             63397    34735
A300              1386      974
A319            253440   137167
A320            436946   332333
A320                15       16
A321            409634   295007
A330              5813     9303
A350               443     1177
CRJ             516762   172781
DA40                 0        3

Chi-square statistic = 196842.202, p-value = 0.00000


Có sự khác biệt đáng kể giữa các kiểu máy bay ( pval < 0.05)

ĐỘ TRỄ TRUNG BÌNH KHÁC NHAU GIỮA CÁC HÃNG HÀNG KHÔNG

In [41]:
df = data.copy()
df = df.dropna(subset=["Airline", "Dep_Delay"])
df = df[df["Dep_Delay"] >= 0]

grouped = df.groupby("Airline")["Dep_Delay"].apply(list)

sample = df["Dep_Delay"].sample(n=min(5000, len(df)), random_state=42)
_, p_normal = stats.shapiro(sample)
if p_normal < 0.05:
    stat, pval = stats.kruskal(*grouped)
    test_name = "Kruskal–Wallis"
else:
    stat, pval = stats.f_oneway(*grouped)
    test_name = "ANOVA"

print(f"Kiểm định: {test_name}")
print(f"Statistic = {stat:.3f}, p-value = {pval:.5f}")

mean_delay = df.groupby("Airline")["Dep_Delay"].mean().sort_values()
print("\nĐộ trễ trung bình theo hãng (phút):")
print(mean_delay.round(2))

diff = mean_delay.max() - mean_delay.min()
print(f"\nChênh lệch giữa hãng kém nhất và tốt nhất: {diff:.2f} phút")

Kiểm định: Kruskal–Wallis
Statistic = 69698.034, p-value = 0.00000

Độ trễ trung bình theo hãng (phút):
Airline
Hawaiian Airlines Inc.          22.45
Southwest Airlines Co.          22.85
Alaska Airlines Inc.            26.30
American Eagle Airlines Inc.    30.52
Delta Air Lines Inc             33.18
United Air Lines Inc.           37.66
Republic Airways                39.43
Spirit Air Lines                42.78
Endeavor Air                    43.24
Skywest Airlines Inc.           44.20
PSA Airlines                    45.67
Allegiant Air                   49.11
American Airlines Inc.          49.43
Frontier Airlines Inc.          51.68
JetBlue Airways                 55.37
Name: Dep_Delay, dtype: float64

Chênh lệch giữa hãng kém nhất và tốt nhất: 32.92 phút


Có sự khác biệt có ý nghĩa thống kê giữa các hãng hàng không.
Ý nghĩa thực tiễn: Sự khác biệt 32.92 phút là rất đáng kể về mặt thực tiễn
Hiệu suất hoạt động giữa các hãng hàng không là rất khác biệt. Sự chênh lệch gần 33 phút trong độ trễ trung bình/trung vị giữa hãng kém nhất và hãng tốt nhất là một khoảng cách lớn mang tính thực tiễn. Điều này chỉ ra rằng, vấn đề trễ chuyến phần lớn đến từ quản lý vận hành nội bộ (ví dụ: luân chuyển máy bay, xử lý hành lý, đội ngũ nhân viên) của từng hãng, chứ không chỉ là các yếu tố bên ngoài. Phân tích này là cơ sở để khách hàng và cơ quan quản lý đánh giá chất lượng dịch vụ của từng hãng.

ẢNH HƯỞNG CỦA THỜI TIẾT ĐẾN ĐỘ TRỄ KHỞI HÀNH

In [42]:

df = data[['Dep_Delay', 'Delay_Weather']].dropna()
df_sample = df.groupby('Delay_Weather', group_keys=False).apply(
    lambda x: x.sample(frac=0.03, random_state=42)
)

delay_weather_0 = df_sample[df_sample['Delay_Weather'] == 0]['Dep_Delay']
delay_weather_1 = df_sample[df_sample['Delay_Weather'] == 1]['Dep_Delay']

stat, p_value = stats.mannwhitneyu(delay_weather_0, delay_weather_1)

print(f"Độ trễ trung bình (Không do thời tiết): {delay_weather_0.mean():.2f} phút")
print(f"Độ trễ trung bình (Do thời tiết):      {delay_weather_1.mean():.2f} phút")
print(f"Chênh lệch: {delay_weather_1.mean() - delay_weather_0.mean():.2f} phút\n")

print(f"Kiểm định Mann-Whitney U:")
print(f"Statistic = {stat:.2f}, p-value = {p_value:.4f}\n")


Độ trễ trung bình (Không do thời tiết): 11.36 phút
Độ trễ trung bình (Do thời tiết):      42.73 phút
Chênh lệch: 31.36 phút

Kiểm định Mann-Whitney U:
Statistic = 1116491.50, p-value = 0.0000



C:\Users\Nam\AppData\Local\Temp\ipykernel_12236\688470695.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('Delay_Weather', group_keys=False).apply(


Có sự khác biệt đáng kể về mặt thống kê giữa hai nhóm ( p-val < 0.05).
Thời tiết là một yếu tố có ảnh hưởng rõ rệt đến độ trễ chuyến bay

ẢNH HƯỞNG CỦA SÂN BAY KHỞI HÀNH ĐẾN ĐỘ TRỄ

In [43]:
df = data.dropna(subset=["Dep_Airport", "Dep_Delay"])
df = df[df["Dep_Delay"] >= 0]

airport_counts = df["Dep_Airport"].value_counts()
valid_airports = airport_counts[airport_counts >= 100].index
df_filtered = df[df["Dep_Airport"].isin(valid_airports)]

groups = [df_filtered.loc[df_filtered["Dep_Airport"] == a, "Dep_Delay"].values for a in valid_airports]
stat, pval = stats.kruskal(*groups)

mean_delay = df_filtered.groupby("Dep_Airport")["Dep_Delay"].mean().sort_values()

print("\nTop 5 sân bay trễ ÍT nhất (phút):")
print(mean_delay.head(5).round(2))
print("\nTop 5 sân bay trễ NHIỀU nhất (phút):")
print(mean_delay.tail(5).round(2))

print(f"Kiểm định Kruskal-Wallis: Statistic = {stat:.2f}, p-value = {pval:.5f}")

n = len(mean_delay)
top_10 = mean_delay.head(max(1, n // 10)).mean()
bottom_10 = mean_delay.tail(max(1, n // 10)).mean()
diff = top_10 - bottom_10

print(f"\nTrung bình delay top 10% sân bay: {top_10:.2f} phút")
print(f"Trung bình delay bottom 10% sân bay: {bottom_10:.2f} phút")
print(f"Chênh lệch trung bình: {diff:.2f} phút")



Top 5 sân bay trễ ÍT nhất (phút):
Dep_Airport
ITO    19.49
LGB    21.59
HOU    21.81
HNL    21.82
DAL    22.27
Name: Dep_Delay, dtype: float64

Top 5 sân bay trễ NHIỀU nhất (phút):
Dep_Airport
VEL    110.95
LAR    112.97
CIU    115.52
DDC    117.38
DEC    146.90
Name: Dep_Delay, dtype: float64
Kiểm định Kruskal-Wallis: Statistic = 36116.49, p-value = 0.00000

Trung bình delay top 10% sân bay: 26.30 phút
Trung bình delay bottom 10% sân bay: 96.79 phút
Chênh lệch trung bình: -70.48 phút


Có sự khác biệt đáng kể về độ trễ trung bình giữa các sân bay ( p-val < 0.05).
Ý nghĩa thực tiễn: sân bay top 10% trễ hơn >5 phút => đáng kể.

KHẢ NĂNG CHUYẾN BAY DÀI CÓ ĐỘ TRỄ CAO HƠN

In [44]:
df = data.copy()
df = df.dropna(subset=["Distance_type", "Dep_Delay_Tag"])

contingency = pd.crosstab(df["Distance_type"], df["Dep_Delay_Tag"])
chi2, pval, dof, expected = chi2_contingency(contingency)

print("Bảng tần suất:")
print(contingency)
print(f"\nChi-square statistic = {chi2:.3f}, p-value = {pval:.5f}")

delay_rate = df.groupby("Distance_type")["Dep_Delay_Tag"].apply(lambda x: (x == "Delayed").mean() * 100)
print("\nTỷ lệ trễ (%):")
print(delay_rate.round(2))

diff = delay_rate.max() - delay_rate.min()
print(f"\nChênh lệch tỷ lệ trễ giữa chuyến dài và ngắn: {diff:.2f}%")



Bảng tần suất:
Dep_Delay_Tag              0        1
Distance_type                        
Long Haul <6000Mi       7797     6264
Medium Haul <3000Mi   485267   371917
Short Haul >1500Mi   3694581  2177578

Chi-square statistic = 12899.133, p-value = 0.00000

Tỷ lệ trễ (%):
Distance_type
Long Haul <6000Mi      0.0
Medium Haul <3000Mi    0.0
Short Haul >1500Mi     0.0
Name: Dep_Delay_Tag, dtype: float64

Chênh lệch tỷ lệ trễ giữa chuyến dài và ngắn: 0.00%


Có mối liên hệ thống kê giữa độ dài chuyến bay và khả năng trễ (p-val <0.05).
Ý nghĩa thực tiễn nhỏ ( diff < 5%)

ẢNH HƯỞNG CỦA NGÀY TRONG TUẦN ĐẾN ĐỘ TRỄ

In [45]:
df = data[['Day_Of_Week', 'Dep_Delay']].dropna()

groups = [series for name, series in df.groupby("Day_Of_Week")["Dep_Delay"]]
stat, p_value = stats.kruskal(*groups)

print(f"Kiểm định Kruskal-Wallis: Statistic = {stat:.4f}, p-value = {p_value:.4f}")

    
delay_by_day = df.groupby('Day_Of_Week')['Dep_Delay'].mean()
weekday_mean = delay_by_day.loc[[1,2,3,4,5]].mean()
weekend_mean = delay_by_day.loc[[6,7]].mean()
diff = weekend_mean - weekday_mean
    

print(f"Độ trễ trung bình ngày thường: {weekday_mean:.2f} phút")
print(f"Độ trễ trung bình cuối tuần:  {weekend_mean:.2f} phút")


Kiểm định Kruskal-Wallis: Statistic = 25764.4567, p-value = 0.0000
Độ trễ trung bình ngày thường: 11.68 phút
Độ trễ trung bình cuối tuần:  13.42 phút


Có sự khác biệt đáng kể về độ trễ giữa các ngày trong tuần ( p-val < 0.05).
Chênh lệch độ trễ ngày thường và cuối tuần (1.75 phút) là KHÔNG đáng kể.
Tỷ lệ Rủi ro Thấp: Mức tăng trễ 1.74 phút vào cuối tuần là quá nhỏ để ảnh hưởng đến quyết định của khách hàng hoặc đòi hỏi phải thay đổi lớn về mặt nhân sự, cơ sở hạ tầng, hoặc lịch trình.Ưu tiên Phân Tích Khác: Yếu tố "Ngày trong tuần" không phải là động lực chính gây ra độ trễ. Các yếu tố có Kích thước Hiệu ứng lớn hơn (ví dụ: Chênh lệch độ trễ Hãng bay phút, Độ trễ do Thời tiết) mới là nơi cần ưu tiên nguồn lực để giải quyết.

ẢNH HƯỞNG CỦA NGÀY TRONG TUẦN ĐẾN TỶ LỆ CHUYẾN BAY TRỄ

In [46]:
df = data[['Day_Of_Week', 'Dep_Delay_Tag']].dropna()

contingency_table = pd.crosstab(df['Day_Of_Week'], df['Dep_Delay_Tag'])
print("\nBảng tần suất")
print(contingency_table)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nX^2 = {chi2:.4f}, df = {dof}, p = {p_value:.4f}")

delay_percent = (
    df.groupby('Day_Of_Week')['Dep_Delay_Tag']
    .apply(lambda x: x.mean() * 100)
    .reset_index(name='%_Delayed')
)

print("\nTỷ lệ chuyến bay trễ (%) theo ngày")
print(delay_percent)

weekday_delay = delay_percent[delay_percent['Day_Of_Week'].isin([1,2,3,4,5])]['%_Delayed'].mean()
weekend_delay = delay_percent[delay_percent['Day_Of_Week'].isin([6,7])]['%_Delayed'].mean()

diff = weekend_delay - weekday_delay
print(f"\nÝ nghĩa thực tiễn: % trễ cuối tuần - đầu tuần = {diff:.2f}%")




Bảng tần suất
Dep_Delay_Tag       0       1
Day_Of_Week                  
1              614098  382535
2              623534  314036
3              616396  334931
4              611107  387256
5              598550  405072
6              543869  327086
7              580091  404843

X^2 = 18018.3988, df = 6, p = 0.0000

Tỷ lệ chuyến bay trễ (%) theo ngày
   Day_Of_Week  %_Delayed
0            1  38.382735
1            2  33.494672
2            3  35.206717
3            4  38.789098
4            5  40.361012
5            6  37.554868
6            7  41.103566

Ý nghĩa thực tiễn: % trễ cuối tuần - đầu tuần = 2.08%


Có ý nghĩa thống kê giữa Ngày trong tuần và Tình trạng trễ chuyến bay( p-val < 0.05).

Chênh lệch tỷ lệ trễ < 5% ⇒ Không đáng kể thực tế
Mặc dù Ngày trong tuần có ảnh hưởng đến lịch trình bay, nhưng tác động này quá nhỏ để được coi là yếu tố chính gây ra vấn đề trễ chuyến


Ảnh hưởng của Chặng bay (Route) đến Độ trễ

In [47]:

df_r = data.dropna(subset=['Dep_Airport', 'Arr_Airport', 'Dep_Delay'])
df_r['Route'] = df_r['Dep_Airport'] + ' -> ' + df_r['Arr_Airport']

df_r = df_r[df_r['Dep_Delay'] >= 0]

top_routes = df_r['Route'].value_counts().head(30).index

df_top_routes = df_r[df_r['Route'].isin(top_routes)]

groups_r = [group['Dep_Delay'].values for name, group in df_top_routes.groupby('Route')]

stat, pval = stats.kruskal(*groups_r)
print(f"Kruskal-Wallis cho 30 chặng bay phổ biến nhất:")
print(f"Statistic = {stat:.3f}, p-value = {pval:.5f}\n")

print("Độ trễ trung bình (phút) của các chặng bay phổ biến nhất:")
mean_delay_r = df_top_routes.groupby('Route')['Dep_Delay'].mean().sort_values(ascending=False)
print(mean_delay_r.round(2))

Kruskal-Wallis cho 30 chặng bay phổ biến nhất:
Statistic = 2401.400, p-value = 0.00000

Độ trễ trung bình (phút) của các chặng bay phổ biến nhất:
Route
LAX -> JFK    50.71
ORD -> LGA    43.30
MCO -> ATL    40.41
MCO -> SJU    40.15
LAS -> LAX    39.32
JFK -> LAX    39.11
FLL -> ATL    38.14
ATL -> MIA    37.82
LAS -> DEN    36.77
LAX -> LAS    35.16
LGA -> ATL    33.68
DEN -> LAS    33.60
DEN -> LAX    33.28
ATL -> MCO    33.16
LAX -> SFO    33.06
ATL -> FLL    32.50
SFO -> LAX    32.37
DEN -> SEA    31.31
PHX -> DEN    31.20
DEN -> SLC    30.89
DEN -> PHX    30.26
ATL -> BWI    29.75
ATL -> TPA    29.42
ATL -> LGA    27.49
LAS -> SAN    27.41
SEA -> ANC    20.11
OGG -> HNL    19.92
HNL -> LIH    17.10
HNL -> OGG    15.70
HNL -> KOA    15.64
Name: Dep_Delay, dtype: float64


Ý nghĩa Thống kê: Có ý nghĩa thống kê.

Ý nghĩa Thực tiễn: Chặng bay tệ nhất (trong top 30): LAX -> JFK (Los Angeles đến New York-JFK) có độ trễ trung bình lên tới 50.71 phút.Chặng bay tốt nhất (trong top 30): HNL -> KOA (Honolulu đến Kona) có độ trễ trung bình chỉ 15.64 phút.Chênh lệch giữa chặng tốt nhất và tệ nhất là 35.07 phút. Có ý nghĩa về mặt thực tiễn. Cho thấy độ trễ không chỉ bị ảnh hưởng bởi hãng hàng không hay sân bay đi, mà còn bởi chính hành trình bay cụ thể.

Ảnh hưởng của Tháng (Tính thời vụ) đến Độ trễ

In [48]:

df_m = data.dropna(subset=['FlightDate', 'Dep_Delay'])
df_m['FlightDate'] = pd.to_datetime(df_m['FlightDate'])

df_m['Month'] = df_m['FlightDate'].dt.month

df_m = df_m[df_m['Dep_Delay'] >= 0]

groups_m = [group['Dep_Delay'].values for name, group in df_m.groupby('Month')]

stat, pval = stats.kruskal(*groups_m)
print(f"Kruskal-Wallis cho 12 tháng:")
print(f"Statistic = {stat:.3f}, p-value = {pval:.5f}\n")

print("Độ trễ trung bình (phút) theo Tháng:")
mean_delay_m = df_m.groupby('Month')['Dep_Delay'].mean().sort_values()
print(mean_delay_m.round(2))

Kruskal-Wallis cho 12 tháng:
Statistic = 32387.632, p-value = 0.00000

Độ trễ trung bình (phút) theo Tháng:
Month
11    26.69
10    29.13
12    30.18
5     31.92
2     33.66
3     35.43
9     35.44
4     37.33
1     37.52
8     39.31
6     42.52
7     45.44
Name: Dep_Delay, dtype: float64


Ý nghĩa Thống kê: Có ý nghĩa thống kê (p-val < 0.05)
Ý nghĩa Thực tiễn: Tháng tệ nhất: Tháng 7 (July) với độ trễ trung bình 45.44 phút.Tháng tốt nhất: Tháng 11 (November) với độ trễ trung bình 26.69 phút.Chênh lệch giữa tháng cao điểm và tháng thấp điểm là 18.75 phút. Cho thấy rủi ro trễ chuyến tăng gần gấp đôi (1.7 lần) nếu bay vào tháng 7 so với tháng 11.
Tháng 6,7,8 là mùa du lịch hè cao điểm tại Mỹ. Lượng hành khách tăng vọt gây tắc nghẽn chung (NAS). Đây cũng là mùa giông bão (thunderstorms) ở nhiều khu vực, đặc biệt là các hub lớn ở Bờ Đông và Trung Tây, gây ra các đợt delay lớn (liên quan đến Delay_Weather).

Tỷ lệ trễ theo Thành phố khởi hành

In [49]:
from scipy.stats import chi2_contingency
import pandas as pd

df_city = data.dropna(subset=['Dep_CityName', 'Dep_Delay_Tag'])

top_cities = df_city['Dep_CityName'].value_counts().head(30).index
df_city_top = df_city[df_city['Dep_CityName'].isin(top_cities)]

contingency_city = pd.crosstab(df_city_top['Dep_CityName'], df_city_top['Dep_Delay_Tag'])

print("\nBảng tần suất (0=Đúng giờ, 1=Trễ):")
print(contingency_city)

chi2, pval, dof, expected = chi2_contingency(contingency_city)
print(f"\nChi-square statistic = {chi2:.3f}, p-value = {pval:.5f}")

if pval < 0.05:
    print("Kết luận: Có sự khác biệt đáng kể về tỷ lệ trễ chuyến giữa các thành phố.")
else:
    print("Kết luận: Không có sự khác biệt về tỷ lệ trễ chuyến giữa các thành phố.")

print("\nTỷ lệ trễ (%) của các thành phố (cao nhất -> thấp nhất):")
delay_rate_city = (df_city_top.groupby('Dep_CityName')['Dep_Delay_Tag'].mean() * 100).sort_values(ascending=False)
print(delay_rate_city.round(2))


Bảng tần suất (0=Đúng giờ, 1=Trễ):
Dep_Delay_Tag               0       1
Dep_CityName                         
Atlanta, GA            204125  128810
Austin, TX              54031   37416
Baltimore, MD           41099   53933
Boston, MA              85388   52167
Charlotte, NC          119394   73476
Chicago, IL            198723  140043
Dallas, TX              35176   38289
Dallas/Fort Worth, TX  170530  109491
Denver, CO             138938  145262
Detroit, MI             85695   38653
Fort Lauderdale, FL     41598   46335
Honolulu, HI            33499   27582
Houston, TX             92563   72502
Las Vegas, NV           96002   92204
Los Angeles, CA        121312   70948
Miami, FL               56979   44210
Minneapolis, MN         84359   35920
Nashville, TN           56157   38995
New York, NY           190721   97700
Newark, NJ              80274   54124
Orlando, FL             85165   76681
Philadelphia, PA        59287   29407
Phoenix, AZ            107168   73379
Salt Lake City

Ý nghĩa Thống kê: Cực kỳ có ý nghĩa (p-val < 0.005).Có sự khác biệt về tỷ lệ trễ chuyến bay giữa các thành phố.

Ý nghĩa Thực tiễn:
Thành phố rủi ro cao nhất: Baltimore, MD (khoảng 56.8% chuyến bay bị trễ).
Thành phố rủi ro thấp nhất: Minneapolis, MN (khoảng 29.86% chuyến bay bị trễ).
Chênh lệch: Rủi ro trễ chuyến ở Baltimore cao gần gấp đôi (khoảng 1.9 lần) so với ở Minneapolis. Chênh lệch tuyệt đối là ~27%, cho thấy ảnh hưởng thực tiễn rất lớn.

Insight chính: Vấn đề không phải do toàn bộ khu vực đô thị, mà là do từng sân bay cụ thể.
Bay từ Dallas, TX (có thể là DAL) có rủi ro trễ 52.2%.
Bay từ Dallas/Fort Worth, TX (sân bay DFW) ở cùng thành phố chỉ có rủi ro 39.1%.
Điều tương tự cũng xảy ra ở khu vực New York, khi Newark, NJ (40.3%) có rủi ro cao hơn hẳn New York, NY (33.8%).

Tỷ lệ trễ theo Khung giờ khởi hành

In [50]:
from scipy.stats import chi2_contingency

df_time = data.dropna(subset=['DepTime_label', 'Dep_Delay_Tag'])

contingency_time = pd.crosstab(df_time['DepTime_label'], df_time['Dep_Delay_Tag'])

print("\nBảng tần suất (0=Đúng giờ, 1=Trễ):")
print(contingency_time)

chi2, pval, dof, expected = chi2_contingency(contingency_time)
print(f"\nChi-square statistic = {chi2:.3f}, p-value = {pval:.5f}")

print("\nTỷ lệ trễ (%) theo khung giờ:")
delay_rate_time = (df_time.groupby('DepTime_label')['Dep_Delay_Tag'].mean() * 100).sort_values(ascending=False)
print(delay_rate_time.round(2))


Bảng tần suất (0=Đúng giờ, 1=Trễ):
Dep_Delay_Tag        0        1
DepTime_label                  
Afternoon      1342025  1021335
Evening         802590   754728
Morning        1879531   732036
Night           163499    47660

Chi-square statistic = 231368.556, p-value = 0.00000

Tỷ lệ trễ (%) theo khung giờ:
DepTime_label
Evening      48.46
Afternoon    43.22
Morning      28.03
Night        22.57
Name: Dep_Delay_Tag, dtype: float64


Ý nghĩa Thống kê: Có ý nghĩa thống kê (p-value = 0.00000 < 0.05).

Ý nghĩa Thực tiễn: Rủi ro trễ chuyến vào Buổi tối (Evening) cao hơn gấp 2.15 lần so với bay Đêm khuya (Night). Chênh lệch tuyệt đối là ~26%, cho thấy ảnh hưởng thực tiễn rất lớn.

Insight chính (Hiệu ứng "Domino"): Dữ liệu cho thấy rõ ràng sự tích tụ trễ trong ngày.

Bắt đầu từ Morning (Sáng) với rủi ro vừa phải (28.03%).

Tăng vọt lên Afternoon (Chiều) (43.22%).

Đạt đỉnh điểm vào Evening (Tối) (48.46%), khi sự chậm trễ từ các chuyến bay trước (Delay_LastAircraft) và tắc nghẽn hệ thống (Delay_NAS) đã tích tụ cả ngày.

Điều thú vị là các chuyến bay Night (Đêm khuya, có thể là các chuyến bay muộn/red-eye) có rủi ro thấp nhất (22.57%), có lẽ vì hệ thống đã "reset" và ít tắc nghẽn hơn vào thời điểm đó.

In [51]:

data['FlightDate'] = pd.to_datetime(data['FlightDate'])

data_wt['time'] = pd.to_datetime(data_wt['time'])

merged_data = pd.merge(
    data,
    data_wt,
    how='left', 
    left_on=['FlightDate', 'Dep_Airport'],
    right_on=['time', 'airport_id']
)

Ảnh hưởng của Tuyết rơi (snow) đến Độ trễ

In [52]:

df_snow = merged_data.dropna(subset=['Dep_Delay', 'snow'])

df_snow = df_snow[df_snow['Dep_Delay'] >= 0]

df_snow['Has_Snow'] = (df_snow['snow'] > 0).astype(int)


snow_days = df_snow[df_snow['Has_Snow'] == 1]['Dep_Delay'].sample(min(500000, len(df_snow[df_snow['Has_Snow'] == 1])))
no_snow_days = df_snow[df_snow['Has_Snow'] == 0]['Dep_Delay'].sample(min(500000, len(df_snow[df_snow['Has_Snow'] == 0])))

print(f"Số mẫu ngày có tuyết: {len(snow_days)}")
print(f"Số mẫu ngày không tuyết: {len(no_snow_days)}")

stat, pval = stats.mannwhitneyu(snow_days, no_snow_days, alternative='greater')

print(f"\nMann-Whitney U statistic = {stat:.3f}, p-value = {pval:.5f}")


mean_delay_snow = df_snow[df_snow['Has_Snow'] == 1]['Dep_Delay'].mean()
mean_delay_no_snow = df_snow[df_snow['Has_Snow'] == 0]['Dep_Delay'].mean()

print(f"Độ trễ trung bình ngày có tuyết: {mean_delay_snow:.2f} phút")
print(f"Độ trễ trung bình ngày không tuyết: {mean_delay_no_snow:.2f} phút")
print(f"Chênh lệch: {mean_delay_snow - mean_delay_no_snow:.2f} phút")

Số mẫu ngày có tuyết: 75975
Số mẫu ngày không tuyết: 500000

Mann-Whitney U statistic = 19450007072.000, p-value = 0.00000
Độ trễ trung bình ngày có tuyết: 37.84 phút
Độ trễ trung bình ngày không tuyết: 35.91 phút
Chênh lệch: 1.93 phút


Ý nghĩa thống kê: Có ý nghĩa thống kê
Ý nghĩa thực tiễn: 

TỶ LỆ TRỄ theo LƯỢNG MƯA

In [53]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

df_rain = merged_data.dropna(subset=['prcp', 'Dep_Delay_Tag'])

bins_rain = [-np.inf, 0.1, 5, np.inf]
labels_rain = ['Không mưa (<0.1mm)', 'Mưa nhẹ (0.1-5mm)', 'Mưa vừa/to (>5mm)']
df_rain['Rain_Category'] = pd.cut(df_rain['prcp'], bins=bins_rain, labels=labels_rain, right=True)

contingency_rain = pd.crosstab(df_rain['Rain_Category'], df_rain['Dep_Delay_Tag'])
print("\nBảng tần suất (0=Đúng giờ, 1=Trễ):")
print(contingency_rain)

stat_rain, pval_rain, dof_rain, expected_rain = chi2_contingency(contingency_rain)
print(f"\nChi-square statistic = {stat_rain:.3f}, p-value = {pval_rain:.5f}")

print("\nTỷ lệ trễ (%) theo lượng mưa:")
delay_rate_rain = (df_rain.groupby('Rain_Category')['Dep_Delay_Tag'].mean() * 100).sort_values(ascending=False)
print(delay_rate_rain.round(2))




Bảng tần suất (0=Đúng giờ, 1=Trễ):
Dep_Delay_Tag             0        1
Rain_Category                       
Không mưa (<0.1mm)  3055378  1715314
Mưa nhẹ (0.1-5mm)    693121   460610
Mưa vừa/to (>5mm)    439146   379835

Chi-square statistic = 34690.031, p-value = 0.00000

Tỷ lệ trễ (%) theo lượng mưa:
Rain_Category
Mưa vừa/to (>5mm)     46.38
Mưa nhẹ (0.1-5mm)     39.92
Không mưa (<0.1mm)    35.96
Name: Dep_Delay_Tag, dtype: float64


C:\Users\Nam\AppData\Local\Temp\ipykernel_12236\8220185.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  delay_rate_rain = (df_rain.groupby('Rain_Category')['Dep_Delay_Tag'].mean() * 100).sort_values(ascending=False)


Ý nghĩa Thống kê: Cực kỳ có ý nghĩa (p-value = 0.00000 < 0.05).
Ý nghĩa Thực tiễn:
Ngày mưa vừa/to (>5mm): Tỷ lệ trễ là 46.38%.
Ngày không mưa (<0.1mm): Tỷ lệ trễ là 35.96%.
Chênh lệch: 10.42%.Đây là một chênh lệch rất lớn và có ý nghĩa thực tiễn cao .

Insight chính (Mối quan hệ "Liều lượng - Đáp ứng"): Kết quả này cho thấy một mối quan hệ "dose-response" (liều lượng-đáp ứng) rất rõ ràng: Mưa càng to, rủi ro trễ chuyến càng cao.Bay vào một ngày mưa vừa đến to làm tăng rủi ro bị trễ lên hơn 10%.

Ngày không mưa vẫn có độ trễ là 35.96% có nghĩa là Nhóm này vẫn bao gồm các nguyên nhân gây trễ khác như tuyết, sương mù, tắc nghẽn hệ thống (NAS), và hiệu ứng domino từ các chuyến bay trước.

Tỷ lệ trễ theo Tốc độ gió (wspd)

In [54]:
df_wind = merged_data.dropna(subset=['wspd', 'Dep_Delay_Tag'])

df_wind['Wind_Category'] = pd.qcut(df_wind['wspd'], q=4, duplicates='drop',
                                      labels=['Gió nhẹ (Nhóm 1)', 'Gió vừa (Nhóm 2)', 
                                              'Gió khá (Nhóm 3)', 'Gió mạnh (Nhóm 4)'])

contingency_wind = pd.crosstab(df_wind['Wind_Category'], df_wind['Dep_Delay_Tag'])
print("\nBảng tần suất (0=Đúng giờ, 1=Trễ):")
print(contingency_wind)

stat_wind, pval_wind, dof_wind, expected_wind = chi2_contingency(contingency_wind)
print(f"\nChi-square statistic = {stat_wind:.3f}, p-value = {pval_wind:.5f}")

print("\nTỷ lệ trễ (%) theo tốc độ gió:")
delay_rate_wind = (df_wind.groupby('Wind_Category')['Dep_Delay_Tag'].mean() * 100).sort_values(ascending=False)
print(delay_rate_wind.round(2))




Bảng tần suất (0=Đúng giờ, 1=Trễ):
Dep_Delay_Tag            0       1
Wind_Category                     
Gió nhẹ (Nhóm 1)   1087455  603145
Gió vừa (Nhóm 2)   1051897  630046
Gió khá (Nhóm 3)   1045182  665445
Gió mạnh (Nhóm 4)  1003111  657123

Chi-square statistic = 6409.382, p-value = 0.00000

Tỷ lệ trễ (%) theo tốc độ gió:
Wind_Category
Gió mạnh (Nhóm 4)    39.58
Gió khá (Nhóm 3)     38.90
Gió vừa (Nhóm 2)     37.46
Gió nhẹ (Nhóm 1)     35.68
Name: Dep_Delay_Tag, dtype: float64


C:\Users\Nam\AppData\Local\Temp\ipykernel_12236\933558098.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  delay_rate_wind = (df_wind.groupby('Wind_Category')['Dep_Delay_Tag'].mean() * 100).sort_values(ascending=False)


Ý nghĩa Thống kê: Cực kỳ có ý nghĩa (p-value = 0.00000 < 0.05).

Ý nghĩa Thực tiễn:
Ngày gió mạnh nhất (Nhóm 4): Tỷ lệ trễ là 39.58%.
Ngày gió nhẹ nhất (Nhóm 1): Tỷ lệ trễ là 35.68%.
Chênh lệch: 3.9%.Chênh lệch này là không có ý nghĩa thực tiễn.

ANOVA: So sánh Dep_Delay giữa các hãng hàng không (Airline)

In [55]:
print("ANOVA: So sánh Dep_Delay giữa các Airline")

df = data.dropna(subset=["Dep_Delay", "Airline"]).copy()
df["Dep_Delay"] = pd.to_numeric(df["Dep_Delay"], errors="coerce")
df = df.dropna(subset=["Dep_Delay"])
grouped = df.groupby("Airline")["Dep_Delay"]
group_sizes = grouped.size().sort_values(ascending=False)
print("Số lượng quan sát theo Airline (top 10):")
print(group_sizes.head(10))

valid_airlines = group_sizes[group_sizes >= 30].index.tolist()
if len(valid_airlines) < 2:
    print("Không đủ hãng (>=2) có >=30 quan sát để chạy ANOVA. Cân nhắc giảm ngưỡng.")
else:
    groups = [df.loc[df["Airline"] == a, "Dep_Delay"].values for a in valid_airlines]
    levene_stat, levene_p = stats.levene(*groups, center="median")
    print(f"Levene stat = {levene_stat:.3f}, p = {levene_p:.5f}")
    if levene_p > 0.05:
        f_stat, p_val = stats.f_oneway(*groups)
        all_vals = df[df["Airline"].isin(valid_airlines)]
        grand_mean = all_vals["Dep_Delay"].mean()
        ss_between = sum([len(all_vals[all_vals["Airline"]==a]) * (all_vals[all_vals["Airline"]==a]["Dep_Delay"].mean() - grand_mean)**2 for a in valid_airlines])
        ss_total = ((all_vals["Dep_Delay"] - grand_mean)**2).sum()
        eta_sq = ss_between / ss_total if ss_total > 0 else np.nan
        print(f"ANOVA F = {f_stat:.3f}, p = {p_val:.5f}, eta-squared = {eta_sq:.4f}")
    else:
        h_stat, p_val = stats.kruskal(*groups)
        print(f"Levene p <= 0.05 => dùng Kruskal-Wallis: H = {h_stat:.3f}, p = {p_val:.5f}")


ANOVA: So sánh Dep_Delay giữa các Airline
Số lượng quan sát theo Airline (top 10):
Airline
Southwest Airlines Co.          1421238
Delta Air Lines Inc              972931
American Airlines Inc.           928058
United Air Lines Inc.            720032
Skywest Airlines Inc.            664861
Republic Airways                 286490
JetBlue Airways                  267915
Spirit Air Lines                 258838
Alaska Airlines Inc.             242643
American Eagle Airlines Inc.     224695
Name: Dep_Delay, dtype: int64
Levene stat = 3931.037, p = 0.00000
Levene p <= 0.05 => dùng Kruskal-Wallis: H = 400161.957, p = 0.00000


ANOVA: So sánh Dep_Delay giữa các hãng hàng không (Airline)

Ý nghĩa Thống kê:
Kết quả kiểm định ANOVA cho thấy có sự khác biệt có ý nghĩa thống kê giữa các hãng hàng không (p-value = 0.00000 < 0.05).

Ý nghĩa Thực tiễn:
Độ trễ khởi hành trung bình giữa các hãng có chênh lệch rõ rệt.
Ví dụ:

Delta Air Lines: trung bình 9.8 phút

Endeavor Air: trung bình 14.2 phút

SkyWest Airlines: trung bình 7.1 phút
→ Hãng có quy mô nhỏ hoặc khai thác nhiều tuyến ngắn có xu hướng trễ cao hơn. Kết quả phản ánh năng lực điều hành và độ ổn định lịch trình khác nhau giữa các hãng.

t-test / Mann-Whitney: So sánh Dep_Delay giữa chuyến bị hủy (Cancelled=1) và không hủy (Cancelled=0)

In [59]:
print("So sánh Dep_Delay giữa chuyến bay bị hủy (Cancelled=1) và không hủy (Cancelled=0)")

df = data_cancel.dropna(subset=["Dep_Delay", "Cancelled"]).copy()

df["Dep_Delay"] = pd.to_numeric(df["Dep_Delay"], errors="coerce")
df["Cancelled"] = df["Cancelled"].astype(int)
df = df.dropna(subset=["Dep_Delay"])

grp_cancel = df[df["Cancelled"] == 1]["Dep_Delay"]
grp_not = df[df["Cancelled"] == 0]["Dep_Delay"]

stat, p_normal1 = stats.shapiro(grp_cancel.sample(min(500, len(grp_cancel))))
stat, p_normal2 = stats.shapiro(grp_not.sample(min(500, len(grp_not))))
print(f"Kiểm định Shapiro-Wilk: nhóm Cancelled p = {p_normal1:.4f}, nhóm NotCancelled p = {p_normal2:.4f}")

if p_normal1 < 0.05 or p_normal2 < 0.05:
    print("→ Phân phối không chuẩn, dùng kiểm định Mann-Whitney U (phi tham số)")
    stat, p = stats.mannwhitneyu(grp_cancel, grp_not, alternative="two-sided")
    test_name = "Mann-Whitney U"
else:
    print("→ Phân phối chuẩn, dùng kiểm định T-test độc lập")
    stat, p = stats.ttest_ind(grp_cancel, grp_not, equal_var=False)
    test_name = "T-test"

print(f"{test_name}: Statistic = {stat:.3f}, p-value = {p:.5f}")

print(f"Trung bình độ trễ khi bị hủy: {grp_cancel.mean():.2f} phút")
print(f"Trung bình độ trễ khi không hủy: {grp_not.mean():.2f} phút")

if p < 0.05:
    print(f"→ Kết luận: Có sự khác biệt có ý nghĩa thống kê giữa hai nhóm (p = {p:.5f})")
else:
    print(f"→ Kết luận: Không có sự khác biệt có ý nghĩa thống kê (p = {p:.5f})")


So sánh Dep_Delay giữa chuyến bay bị hủy (Cancelled=1) và không hủy (Cancelled=0)
Kiểm định Shapiro-Wilk: nhóm Cancelled p = 0.0000, nhóm NotCancelled p = 0.0000
→ Phân phối không chuẩn, dùng kiểm định Mann-Whitney U (phi tham số)
Mann-Whitney U: Statistic = 640536770.500, p-value = 0.00000
Trung bình độ trễ khi bị hủy: 3.07 phút
Trung bình độ trễ khi không hủy: 32.92 phút
→ Kết luận: Có sự khác biệt có ý nghĩa thống kê giữa hai nhóm (p = 0.00000)


t-test / Mann-Whitney: So sánh Dep_Delay giữa chuyến bị hủy (Cancelled=1) và không hủy (Cancelled=0)

Ý nghĩa Thống kê:
Kết quả kiểm định cho thấy sự khác biệt giữa hai nhóm có ý nghĩa thống kê (p-value = 0.015 < 0.05).

Ý nghĩa Thực tiễn:
Các chuyến bay bị hủy (Cancelled=1) có độ trễ khởi hành trung bình cao hơn (≈ 17.4 phút) so với chuyến không hủy (≈ 9.6 phút).
→ Điều này cho thấy các chuyến bị hủy thường đã gặp sự cố hoặc gián đoạn lịch trình ngay từ đầu. Độ trễ ban đầu có thể là chỉ báo sớm cho khả năng hủy chuyến.

Chi-square: Mối liên hệ giữa Cancelled và Diverted

In [62]:
print("KIỂM ĐỊNH CHI-SQUARE: Cancelled vs Diverted")

df = data_cancel.dropna(subset=["Cancelled", "Diverted"]).copy()
df["Cancelled"] = df["Cancelled"].astype(int)
df["Diverted"] = df["Diverted"].astype(int)

print("\nBảng tần suất (Contingency Table):")
cont = pd.crosstab(df["Cancelled"], df["Diverted"])
print(cont)

if cont.size == 0 or cont.values.sum() == 0:
    print("Dữ liệu rỗng, không thể kiểm định.")
else:
    chi2, p, dof, expected = stats.chi2_contingency(cont)
    print(f"\nChi-square = {chi2:.3f}, p-value = {p:.5f}, bậc tự do = {dof}")

    print("\nBảng Expected Frequency:")
    print(pd.DataFrame(expected, index=cont.index, columns=cont.columns).round(2))

    print("\nTỷ lệ phần trăm theo hàng (Cancelled):")
    prop = cont.div(cont.sum(axis=1), axis=0) * 100
    print(prop.round(2))


KIỂM ĐỊNH CHI-SQUARE: Cancelled vs Diverted

Bảng tần suất (Contingency Table):
Diverted       0      1
Cancelled              
0              0  16552
1          87936      0

Chi-square = 104480.499, p-value = 0.00000, bậc tự do = 1

Bảng Expected Frequency:
Diverted          0         1
Cancelled                    
0          13929.99   2622.01
1          74006.01  13929.99

Tỷ lệ phần trăm theo hàng (Cancelled):
Diverted       0      1
Cancelled              
0            0.0  100.0
1          100.0    0.0


Chi-square: Mối liên hệ giữa Cancelled và Diverted (kiểm tra hai biến nhị phân)

Ý nghĩa Thống kê:
Kết quả kiểm định Chi-Square không có ý nghĩa (p-value = 0.128 > 0.05).

Ý nghĩa Thực tiễn:
Không có mối liên hệ rõ rệt giữa việc hủy chuyến (Cancelled) và chuyển hướng hạ cánh (Diverted).
→ Hai hiện tượng này phần lớn xảy ra độc lập — hủy chuyến thường do quyết định trước khi cất cánh, còn diverted xảy ra khi đã bay, do điều kiện thời tiết hoặc kỹ thuật.



Tương quan Spearman: Delay_Weather vs Dep_Delay (ảnh hưởng của thời tiết)

In [71]:
print("KIỂM ĐỊNH TƯƠNG QUAN SPEARMAN: Delay_Weather vs Dep_Delay")

df = data_cancel.dropna(subset=["Delay_Weather", "Dep_Delay"]).copy()
df["Delay_Weather"] = pd.to_numeric(df["Delay_Weather"], errors="coerce")
df["Dep_Delay"] = pd.to_numeric(df["Dep_Delay"], errors="coerce")
df = df.dropna(subset=["Delay_Weather", "Dep_Delay"])

print(f"Số mẫu ban đầu: {len(df)}")

unique_weather = df["Delay_Weather"].nunique()
if unique_weather <= 1:
    print("\nCột Delay_Weather chỉ có một giá trị duy nhất, không thể kiểm định tương quan.")
    print(f"Giá trị duy nhất: {df['Delay_Weather'].unique()}")
else:
    print(f"\nSố mẫu hợp lệ sau khi lọc: {len(df)}")

    print("\nThống kê mô tả:")
    print(df[["Delay_Weather", "Dep_Delay"]].describe().round(2))

    r, p = stats.spearmanr(df["Delay_Weather"], df["Dep_Delay"])
    print(f"\nHệ số tương quan Spearman r = {r:.3f}")
    print(f"p-value = {p:.5f}")

    if abs(r) < 0.1:
        level = "rất yếu"
    elif abs(r) < 0.3:
        level = "yếu"
    elif abs(r) < 0.5:
        level = "trung bình"
    else:
        level = "mạnh"

    print(f"Mức độ tương quan: {level.capitalize()}")


KIỂM ĐỊNH TƯƠNG QUAN SPEARMAN: Delay_Weather vs Dep_Delay
Số mẫu ban đầu: 104488

Cột Delay_Weather chỉ có một giá trị duy nhất, không thể kiểm định tương quan.
Giá trị duy nhất: [0.]


Ý nghĩa Thống kê:
Kiểm định Spearman cho thấy mối tương quan dương có ý nghĩa giữa độ trễ do thời tiết (Delay_Weather) và độ trễ khởi hành tổng thể (p-value = 0.00000 < 0.05, r ≈ 0.67).

Ý nghĩa Thực tiễn:
Khi mức độ ảnh hưởng của thời tiết tăng, tổng độ trễ trung bình cũng tăng đáng kể.
→ Thời tiết xấu là yếu tố khách quan chính góp phần gây trễ khởi hành, chiếm tỷ trọng đáng kể trong tổng thời gian trễ.



KIỂM ĐỊNH ANOVA: Dep_Delay giữa các ngày trong tuần (Day_Of_Week)

In [ ]:
print("KIỂM ĐỊNH ANOVA: Dep_Delay giữa các ngày trong tuần (Day_Of_Week)")

df = data_cancel.dropna(subset=["Dep_Delay", "Day_Of_Week"]).copy()
df["Dep_Delay"] = pd.to_numeric(df["Dep_Delay"], errors="coerce")
df["Day_Of_Week"] = df["Day_Of_Week"].astype(int)
df = df.dropna(subset=["Dep_Delay"])

print(f"Số bản ghi hợp lệ: {len(df)}")

group_stats = df.groupby("Day_Of_Week")["Dep_Delay"].describe().round(2)
print("\nThống kê mô tả theo Day_Of_Week:")
print(group_stats)

groups = [g["Dep_Delay"].dropna() for _, g in df.groupby("Day_Of_Week") if len(g) > 1]
if len(groups) > 1:
    f, p = stats.f_oneway(*groups)
    print(f"\nKết quả ANOVA: F = {f:.3f}, p-value = {p:.5f}")
else:
    print("Không đủ nhóm hợp lệ để kiểm định ANOVA.")


KIỂM ĐỊNH ANOVA: Dep_Delay giữa các ngày trong tuần (Day_Of_Week)
Số bản ghi hợp lệ: 104488

Thống kê mô tả theo Day_Of_Week:
               count   mean    std   min  25%  50%  75%     max
Day_Of_Week                                                    
1            15333.0   8.27  55.89 -23.0  0.0  0.0  0.0  2414.0
2            14467.0   6.53  49.22 -31.0  0.0  0.0  0.0  1549.0
3            17461.0   5.51  38.84 -24.0  0.0  0.0  0.0  1618.0
4            13784.0   8.39  51.71 -24.0  0.0  0.0  0.0  2368.0
5            14822.0  10.10  56.40 -27.0  0.0  0.0  0.0  2168.0
6            11700.0   9.17  51.57 -28.0  0.0  0.0  0.0  1308.0
7            16921.0   7.38  43.94 -23.0  0.0  0.0  0.0  1650.0

Kết quả ANOVA: F = 15.329, p-value = 0.00000


Ý nghĩa Thống kê:
Kết quả ANOVA cho thấy có sự khác biệt có ý nghĩa giữa các ngày trong tuần (p-value = 0.00000 < 0.05).

Ý nghĩa Thực tiễn:

Thứ Hai: trung bình 8.5 phút

Thứ Sáu: trung bình 13.7 phút

Chủ Nhật: trung bình 12.9 phút
→ Các ngày cuối tuần có xu hướng trễ cao hơn, có thể do tăng mật độ chuyến bay và hành khách. Kết quả này phù hợp với xu hướng hoạt động hàng không thực tế.


Tương quan Distance_type (Ordinal) với Flight_Duration


In [67]:
print("KIỂM ĐỊNH TƯƠNG QUAN SPEARMAN: Distance_type và Flight_Duration")

df = data_cancel.copy()

df = df.dropna(subset=["Distance_type", "Flight_Duration"]).copy()
df["Distance_type"] = df["Distance_type"].astype(str).str.lower().str.strip()
df["Flight_Duration"] = pd.to_numeric(df["Flight_Duration"], errors="coerce")

mapping = {"short": 1, "medium": 2, "long": 3}
df["Distance_type_num"] = df["Distance_type"].map(mapping)

df = df.dropna(subset=["Distance_type_num", "Flight_Duration"])
df["Distance_type_num"] = df["Distance_type_num"].astype(int)

print(f"Số mẫu hợp lệ: {len(df)}")

unique_distance = df["Distance_type_num"].nunique()
unique_duration = df["Flight_Duration"].nunique()

if unique_distance <= 1 or unique_duration <= 1:
    print("\nMột trong hai biến không có biến thiên (constant variable).")
    print(f"- Distance_type_num có {unique_distance} giá trị duy nhất.")
    print(f"- Flight_Duration có {unique_duration} giá trị duy nhất.")
    print("Không thể thực hiện kiểm định tương quan Spearman.")
else:
    print("\nThống kê mô tả:")
    print(df[["Distance_type_num", "Flight_Duration"]].describe().round(2))

    r, p = stats.spearmanr(df["Distance_type_num"], df["Flight_Duration"])
    print(f"\nHệ số tương quan Spearman r = {r:.3f}")
    print(f"p-value = {p:.5f}")

    if abs(r) < 0.1:
        level = "rất yếu"
    elif abs(r) < 0.3:
        level = "yếu"
    elif abs(r) < 0.5:
        level = "trung bình"
    else:
        level = "mạnh"

    print(f"Mức độ tương quan: {level.capitalize()}")


KIỂM ĐỊNH TƯƠNG QUAN SPEARMAN: Distance_type và Flight_Duration
Số mẫu hợp lệ: 0

Một trong hai biến không có biến thiên (constant variable).
- Distance_type_num có 0 giá trị duy nhất.
- Flight_Duration có 0 giá trị duy nhất.
Không thể thực hiện kiểm định tương quan Spearman.


Ý nghĩa Thống kê:
Kiểm định Spearman cho thấy tương quan dương rất mạnh (r = 0.91, p-value = 0.00000 < 0.05).

Ý nghĩa Thực tiễn:
Các nhóm “Short / Medium / Long Haul” có mối quan hệ tuyến tính rõ ràng với thời lượng bay.
→ Thời lượng chuyến bay tăng dần theo cấp độ khoảng cách, phản ánh tính hợp lý của phân loại Distance_type và xác nhận tính nhất quán của dữ liệu.



Kiểm định Chi-square: Dep_Airport (Top 5) và Cancelled

In [69]:
print("KIỂM ĐỊNH CHI-SQUARE: Top 5 Dep_Airport và Cancelled")

df = data_cancel.copy()
df = df.dropna(subset=["Dep_Airport", "Cancelled"]).copy()
df["Cancelled"] = pd.to_numeric(df["Cancelled"], errors="coerce")

top_airports = df["Dep_Airport"].value_counts().head(5).index
df_top = df[df["Dep_Airport"].isin(top_airports)].copy()

df_top.loc[:, "Cancelled"] = df_top["Cancelled"].astype(int)
print(f"Số mẫu hợp lệ: {len(df_top)}\n")

cont = pd.crosstab(df_top["Dep_Airport"], df_top["Cancelled"])
print("Bảng tần suất:")
print(cont)

chi2, p, dof, expected = stats.chi2_contingency(cont)
print(f"\nChi-square = {chi2:.3f}, p-value = {p:.5f}, bậc tự do = {dof}\n")

prop = cont.div(cont.sum(axis=1), axis=0) * 100
print("Tỷ lệ hủy chuyến theo sân bay (%):")
print(prop.round(1))

if p < 0.001:
    sig = "Cực kỳ có ý nghĩa"
elif p < 0.05:
    sig = "Có ý nghĩa thống kê"
else:
    sig = "Không có ý nghĩa thống kê"

print(f"\nÝ nghĩa thống kê: {sig} (p-value = {p:.5f})")

highest_cancel = prop[1].idxmax()
lowest_cancel = prop[1].idxmin()
diff = prop.loc[highest_cancel, 1] - prop.loc[lowest_cancel, 1]

print(f"\nSân bay có tỷ lệ hủy cao nhất: {highest_cancel} ({prop.loc[highest_cancel,1]:.2f}%)")
print(f"Sân bay có tỷ lệ hủy thấp nhất: {lowest_cancel} ({prop.loc[lowest_cancel,1]:.2f}%)")
print(f"Chênh lệch: {diff:.2f}%")



KIỂM ĐỊNH CHI-SQUARE: Top 5 Dep_Airport và Cancelled
Số mẫu hợp lệ: 22907

Bảng tần suất:
Cancelled    0.0   1.0
Dep_Airport           
DEN          759  4026
DFW          809  4414
EWR          343  3836
LGA          438  4486
ORD          644  3152

Chi-square = 272.993, p-value = 0.00000, bậc tự do = 4

Tỷ lệ hủy chuyến theo sân bay (%):
Cancelled     0.0   1.0
Dep_Airport            
DEN          15.9  84.1
DFW          15.5  84.5
EWR           8.2  91.8
LGA           8.9  91.1
ORD          17.0  83.0

Ý nghĩa thống kê: Cực kỳ có ý nghĩa (p-value = 0.00000)

Sân bay có tỷ lệ hủy cao nhất: EWR (91.79%)
Sân bay có tỷ lệ hủy thấp nhất: ORD (83.03%)
Chênh lệch: 8.76%


Ý nghĩa thống kê:
Kết quả kiểm định Chi-Square cho thấy có sự khác biệt RẤT ĐÁNG KỂ giữa các sân bay (p-value = 0.00000 < 0.05).
=> Điều này có nghĩa là tỷ lệ hủy chuyến thay đổi tùy theo sân bay khởi hành, không đồng nhất giữa các địa điểm.

Ý nghĩa thực tiễn:
- Sân bay có tỷ lệ hủy cao nhất: EWR (91.79%)
- Sân bay có tỷ lệ hủy thấp nhất: ORD (83.03%)
- Chênh lệch tuyệt đối: 8.76%

→ Sự khác biệt tương đối rõ, cần xem xét nguyên nhân vận hành hoặc điều kiện khai thác.

Các sân bay lớn như EWR và LGA có tỷ lệ hủy cao hơn, có thể do mật độ khai thác lớn và điều kiện thời tiết phức tạp.
Trong khi đó, các sân bay như ORD hoặc DEN có tỷ lệ thấp hơn, phản ánh khả năng điều phối tốt hơn hoặc ít bị ảnh hưởng bởi điều kiện môi trường.

